# Journal Extension J6 — Calibration, Reliability & Selective Prediction
## Original 83.95% Thesis Protocol

This notebook evaluates whether the frozen proposed model is not only accurate, but also **reliable when expressing confidence**.

No model training is performed.

## Analyses

### Calibration
For:
- Final proposed ensemble
- Final proposed ensemble + validation-only scalar temperature scaling
- SimCLR No-TTA
- DINOv2 ViT-S/14 3-seed mean
- Optimized Weighted Ensemble

Metrics:
- Expected Calibration Error (ECE, 15 equal-width bins)
- Adaptive ECE (15 equal-frequency bins)
- Maximum Calibration Error (MCE)
- Multiclass Brier score
- Negative Log-Likelihood (NLL)
- mean confidence
- accuracy
- confidence-accuracy gap

### Reliability diagrams
- Final proposed model
- Final + validation-only temperature scaling
- SimCLR
- DINOv2
- Weighted Ensemble

### Validation-only post-hoc temperature scaling
A **single scalar temperature** is selected on the validation set by minimizing NLL.
It is then applied once to the held-out test probabilities.

Important:
- scalar temperature scaling does **not change argmax predictions**,
- therefore it does not change Accuracy/BACC,
- it is used only to improve probability calibration.

### Selective prediction
At retained coverage:
- 100%
- 90%
- 80%
- 70%
- 60%
- 50%

The notebook reports:
- confidence threshold
- retained samples
- accuracy
- balanced accuracy
- macro-F1
- error risk
- class counts at each coverage

It also produces a risk-coverage curve and AURC.

## Required Kaggle inputs

1. **Original full thesis pipeline output** (`last-part-of-the-previous-one`)
   - for SimCLR No-TTA validation/test probabilities.

2. **Confidence-gated ensemble output**
   - for frozen final and optimized weighted validation/test probabilities.

3. **Executed J2 output**
   - for same-split DINOv2 validation/test probabilities.

4. J1 is optional for additional protocol verification.

## No images, GPU, or training are required.

In [1]:
# ============================================================
# CELL 1 — IMPORTS / CONFIG
# ============================================================
import json
import math
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)

KAGGLE_INPUT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")

OUTPUT_DIR = WORKING / "j6_original_split_calibration"
TABLE_DIR = OUTPUT_DIR / "tables"
FIG_DIR = OUTPUT_DIR / "figures"
AUDIT_DIR = OUTPUT_DIR / "audit"

for d in [OUTPUT_DIR, TABLE_DIR, FIG_DIR, AUDIT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["MEL","NV","BCC","AK","BKL","DF","VASC","SCC"]
NUM_CLASSES = 8

EXPECTED_VAL_N = 2533
EXPECTED_TEST_N = 2534

N_BINS = 15
COVERAGES = [1.00, 0.90, 0.80, 0.70, 0.60, 0.50]

print("=" * 96)
print("J6 — CALIBRATION / RELIABILITY / SELECTIVE PREDICTION")
print("=" * 96)
print("Validation N:", EXPECTED_VAL_N)
print("Test N      :", EXPECTED_TEST_N)
print("ECE bins    :", N_BINS)
print("Coverages   :", COVERAGES)
print("Output      :", OUTPUT_DIR)

J6 — CALIBRATION / RELIABILITY / SELECTIVE PREDICTION
Validation N: 2533
Test N      : 2534
ECE bins    : 15
Coverages   : [1.0, 0.9, 0.8, 0.7, 0.6, 0.5]
Output      : /kaggle/working/j6_original_split_calibration


In [2]:
# ============================================================
# CELL 2 — HELPERS
# ============================================================

def find_all(filename):
    return sorted(
        KAGGLE_INPUT.rglob(filename),
        key=lambda p: (len(p.parts), str(p))
    )

def normalize_probabilities(probabilities):
    p = np.asarray(probabilities, dtype=np.float64)

    if p.ndim != 2 or p.shape[1] != NUM_CLASSES:
        raise ValueError(
            f"Expected probability shape (N,{NUM_CLASSES}); got {p.shape}"
        )

    p = np.nan_to_num(p, nan=0.0, posinf=0.0, neginf=0.0)
    p = np.clip(p, 1e-12, None)
    p /= p.sum(axis=1, keepdims=True)

    return p.astype(np.float64)

def multiclass_nll(y_true, probabilities):
    p = normalize_probabilities(probabilities)
    return float(
        -np.mean(
            np.log(
                p[np.arange(len(y_true)), y_true]
            )
        )
    )

def multiclass_brier(y_true, probabilities):
    p = normalize_probabilities(probabilities)

    one_hot = np.zeros_like(p)
    one_hot[
        np.arange(len(y_true)),
        y_true
    ] = 1.0

    return float(
        np.mean(
            np.sum(
                (p - one_hot) ** 2,
                axis=1,
            )
        )
    )

def reliability_equal_width(y_true, probabilities, n_bins=N_BINS):
    p = normalize_probabilities(probabilities)
    pred = np.argmax(p, axis=1)
    conf = np.max(p, axis=1)
    correct = (pred == y_true).astype(np.float64)

    edges = np.linspace(0.0, 1.0, n_bins + 1)

    rows = []

    for i in range(n_bins):
        lo = edges[i]
        hi = edges[i + 1]

        if i == n_bins - 1:
            mask = (conf >= lo) & (conf <= hi)
        else:
            mask = (conf >= lo) & (conf < hi)

        n = int(np.sum(mask))

        if n == 0:
            rows.append({
                "bin": i + 1,
                "lower": lo,
                "upper": hi,
                "n": 0,
                "mean_confidence": np.nan,
                "accuracy": np.nan,
                "gap": np.nan,
            })
            continue

        mean_conf = float(np.mean(conf[mask]))
        acc = float(np.mean(correct[mask]))

        rows.append({
            "bin": i + 1,
            "lower": lo,
            "upper": hi,
            "n": n,
            "mean_confidence": mean_conf,
            "accuracy": acc,
            "gap": abs(acc - mean_conf),
        })

    return pd.DataFrame(rows)

def ece_equal_width(y_true, probabilities, n_bins=N_BINS):
    table = reliability_equal_width(
        y_true,
        probabilities,
        n_bins=n_bins,
    )

    n_total = len(y_true)

    valid = table["n"] > 0

    ece = float(
        np.sum(
            (
                table.loc[valid, "n"]
                / n_total
            )
            * table.loc[valid, "gap"]
        )
    )

    mce = float(
        table.loc[valid, "gap"].max()
    )

    return ece, mce, table

def adaptive_ece(y_true, probabilities, n_bins=N_BINS):
    p = normalize_probabilities(probabilities)
    pred = np.argmax(p, axis=1)
    conf = np.max(p, axis=1)
    correct = (pred == y_true).astype(np.float64)

    order = np.argsort(conf)

    bins = np.array_split(
        order,
        n_bins,
    )

    weighted_gap = 0.0

    rows = []

    for i, idx in enumerate(bins):
        if len(idx) == 0:
            continue

        c = float(np.mean(conf[idx]))
        a = float(np.mean(correct[idx]))
        gap = abs(a - c)

        weighted_gap += (
            len(idx)
            / len(y_true)
        ) * gap

        rows.append({
            "bin": i + 1,
            "n": len(idx),
            "min_confidence": float(np.min(conf[idx])),
            "max_confidence": float(np.max(conf[idx])),
            "mean_confidence": c,
            "accuracy": a,
            "gap": gap,
        })

    return float(weighted_gap), pd.DataFrame(rows)

def calibration_metrics(y_true, probabilities):
    p = normalize_probabilities(probabilities)
    pred = np.argmax(p, axis=1)
    conf = np.max(p, axis=1)

    acc = float(
        accuracy_score(
            y_true,
            pred,
        )
    )

    ece, mce, _ = ece_equal_width(
        y_true,
        p,
        n_bins=N_BINS,
    )

    aece, _ = adaptive_ece(
        y_true,
        p,
        n_bins=N_BINS,
    )

    mean_conf = float(
        np.mean(conf)
    )

    return {
        "accuracy": acc,
        "mean_confidence": mean_conf,
        "confidence_accuracy_gap": mean_conf - acc,
        "ece_15": ece,
        "adaptive_ece_15": aece,
        "mce_15": mce,
        "nll": multiclass_nll(y_true, p),
        "brier": multiclass_brier(y_true, p),
    }

def temperature_scale(probabilities, temperature):
    p = normalize_probabilities(probabilities)

    log_p = np.log(
        np.clip(
            p,
            1e-12,
            1.0,
        )
    )

    scaled = log_p / float(temperature)
    scaled -= scaled.max(axis=1, keepdims=True)

    return normalize_probabilities(
        np.exp(scaled)
    )

print("Helpers ready.")

Helpers ready.


In [3]:
# ============================================================
# CELL 3 — LOAD FROZEN FINAL + WEIGHTED ENSEMBLE
# ============================================================

hits = find_all(
    "final_internal_probabilities.npz"
)

ranked = sorted(
    hits,
    key=lambda p: (
        0 if "reuse_only_all_model_ensemble" in str(p) else
        1 if "j1_original_protocol_lock" in str(p) else
        9,
        len(p.parts),
        str(p),
    )
)

FINAL_NPZ = None

for p in ranked:
    try:
        with np.load(p) as z:
            required = {
                "y_val",
                "y_test",
                "final_val_prob",
                "final_test_prob",
                "weighted_val_prob",
                "weighted_test_prob",
            }

            if not required.issubset(z.files):
                continue

            if len(z["y_val"]) != EXPECTED_VAL_N:
                continue

            if len(z["y_test"]) != EXPECTED_TEST_N:
                continue

        FINAL_NPZ = p
        break

    except Exception:
        continue

if FINAL_NPZ is None:
    raise FileNotFoundError(
        "Original 2533/2534 final_internal_probabilities.npz not found."
    )

with np.load(FINAL_NPZ) as z:
    Y_VAL = np.asarray(
        z["y_val"],
        dtype=np.int32,
    )

    Y_TEST = np.asarray(
        z["y_test"],
        dtype=np.int32,
    )

    FINAL_VAL = normalize_probabilities(
        z["final_val_prob"]
    )

    FINAL_TEST = normalize_probabilities(
        z["final_test_prob"]
    )

    WEIGHTED_VAL = normalize_probabilities(
        z["weighted_val_prob"]
    )

    WEIGHTED_TEST = normalize_probabilities(
        z["weighted_test_prob"]
    )

print("Final ensemble source:")
print(FINAL_NPZ)
print("VAL / TEST:", len(Y_VAL), len(Y_TEST))

Final ensemble source:
/kaggle/input/notebooks/anirbanroysudipto/confidence-gated-ensemble/reuse_only_all_model_ensemble/final_internal_probabilities.npz
VAL / TEST: 2533 2534


In [4]:
# ============================================================
# CELL 4 — LOAD SIMCLR NO-TTA
# ============================================================

sim_hits = find_all(
    "multiseed_probabilities_no_tta.npz"
)

SIMCLR_NPZ = None

for p in sim_hits:
    try:
        with np.load(p) as z:
            if not {
                "y_val",
                "y_test",
                "val_prob",
                "test_prob",
            }.issubset(z.files):
                continue

            yv = np.asarray(
                z["y_val"],
                dtype=np.int32,
            )

            yt = np.asarray(
                z["y_test"],
                dtype=np.int32,
            )

            if not np.array_equal(
                yv,
                Y_VAL,
            ):
                continue

            if not np.array_equal(
                yt,
                Y_TEST,
            ):
                continue

        SIMCLR_NPZ = p
        break

    except Exception:
        continue

if SIMCLR_NPZ is None:
    raise FileNotFoundError(
        "Original SimCLR No-TTA probability file not found."
    )

with np.load(SIMCLR_NPZ) as z:
    SIMCLR_VAL = normalize_probabilities(
        z["val_prob"]
    )

    SIMCLR_TEST = normalize_probabilities(
        z["test_prob"]
    )

print("SimCLR source:")
print(SIMCLR_NPZ)

SimCLR source:
/kaggle/input/notebooks/anirbanroysudipto/last-part-of-the-previous-one/rotation_ssl_paperlike/simclr_finetune_multiseed_bacc_v2/multiseed_probabilities_no_tta.npz


In [5]:
# ============================================================
# CELL 5 — LOAD DINOv2 MULTI-SEED SAME-SPLIT PROBABILITIES
# ============================================================

dino_hits = find_all(
    "dinov2_original_split_multiseed_probabilities.npz"
)

DINO_NPZ = None

for p in dino_hits:
    try:
        with np.load(p, allow_pickle=True) as z:
            if not {
                "y_val",
                "y_test",
                "val_prob",
                "test_prob",
            }.issubset(z.files):
                continue

            yv = np.asarray(
                z["y_val"],
                dtype=np.int32,
            )

            yt = np.asarray(
                z["y_test"],
                dtype=np.int32,
            )

            if not np.array_equal(
                yv,
                Y_VAL,
            ):
                continue

            if not np.array_equal(
                yt,
                Y_TEST,
            ):
                continue

        DINO_NPZ = p
        break

    except Exception:
        continue

if DINO_NPZ is None:
    raise FileNotFoundError(
        "J2 DINO same-split probability file not found."
    )

with np.load(
    DINO_NPZ,
    allow_pickle=True,
) as z:
    DINO_VAL = normalize_probabilities(
        np.asarray(
            z["val_prob"],
            dtype=np.float64,
        )
    )

    DINO_TEST = normalize_probabilities(
        np.asarray(
            z["test_prob"],
            dtype=np.float64,
        )
    )

print("DINO source:")
print(DINO_NPZ)

DINO source:
/kaggle/input/notebooks/anirbanroysudipto/journal-extension-j2/j2_dinov2_original_split/dinov2_original_split_multiseed_probabilities.npz


In [6]:
# ============================================================
# CELL 6 — VALIDATION-ONLY SCALAR TEMPERATURE SELECTION
# ============================================================

# Coarse + fine deterministic grid search.
coarse_grid = np.linspace(
    0.25,
    5.00,
    191,
)

coarse_nll = np.array([
    multiclass_nll(
        Y_VAL,
        temperature_scale(
            FINAL_VAL,
            t,
        ),
    )
    for t in coarse_grid
])

coarse_best = float(
    coarse_grid[
        np.argmin(coarse_nll)
    ]
)

fine_low = max(
    0.05,
    coarse_best - 0.10,
)

fine_high = coarse_best + 0.10

fine_grid = np.linspace(
    fine_low,
    fine_high,
    401,
)

fine_nll = np.array([
    multiclass_nll(
        Y_VAL,
        temperature_scale(
            FINAL_VAL,
            t,
        ),
    )
    for t in fine_grid
])

BEST_TEMPERATURE = float(
    fine_grid[
        np.argmin(fine_nll)
    ]
)

FINAL_VAL_TEMP = temperature_scale(
    FINAL_VAL,
    BEST_TEMPERATURE,
)

FINAL_TEST_TEMP = temperature_scale(
    FINAL_TEST,
    BEST_TEMPERATURE,
)

raw_val_nll = multiclass_nll(
    Y_VAL,
    FINAL_VAL,
)

scaled_val_nll = multiclass_nll(
    Y_VAL,
    FINAL_VAL_TEMP,
)

print("Validation-only selected temperature:", BEST_TEMPERATURE)
print("Raw validation NLL               :", raw_val_nll)
print("Temperature-scaled validation NLL:", scaled_val_nll)

# Scalar temperature must preserve class predictions.
if not np.array_equal(
    np.argmax(FINAL_TEST, axis=1),
    np.argmax(FINAL_TEST_TEMP, axis=1),
):
    raise RuntimeError(
        "Scalar temperature unexpectedly changed argmax predictions."
    )

temperature_audit = {
    "selection_split": "validation only",
    "test_used_for_temperature_selection": False,
    "selected_temperature": BEST_TEMPERATURE,
    "raw_validation_nll": raw_val_nll,
    "scaled_validation_nll": scaled_val_nll,
    "test_argmax_preserved": True,
}

with open(
    AUDIT_DIR / "validation_temperature_selection.json",
    "w",
) as f:
    json.dump(
        temperature_audit,
        f,
        indent=2,
    )

print("Temperature-selection audit: PASSED")

Validation-only selected temperature: 0.866
Raw validation NLL               : 0.4789775446151375
Temperature-scaled validation NLL: 0.47181396066315745
Temperature-selection audit: PASSED


In [7]:
# ============================================================
# CELL 7 — CALIBRATION METRICS
# ============================================================

MODEL_PROB = {
    "Final Proposed": FINAL_TEST,
    "Final Proposed + Val-Temp": FINAL_TEST_TEMP,
    "SimCLR No-TTA": SIMCLR_TEST,
    "DINOv2 ViT-S/14": DINO_TEST,
    "Optimized Weighted Ensemble": WEIGHTED_TEST,
}

rows = []

for model_name, prob in MODEL_PROB.items():
    m = calibration_metrics(
        Y_TEST,
        prob,
    )

    rows.append({
        "model": model_name,
        **m,
    })

CALIBRATION_DF = pd.DataFrame(
    rows
)

display(
    CALIBRATION_DF
)

CALIBRATION_DF.to_csv(
    TABLE_DIR / "test_calibration_metrics.csv",
    index=False,
)

# Percent-formatted companion table for ECE/MCE/gaps.
CALIBRATION_PERCENT_DF = CALIBRATION_DF.copy()

for c in [
    "accuracy",
    "mean_confidence",
    "confidence_accuracy_gap",
    "ece_15",
    "adaptive_ece_15",
    "mce_15",
]:
    CALIBRATION_PERCENT_DF[c] = (
        100.0
        * CALIBRATION_PERCENT_DF[c]
    )

CALIBRATION_PERCENT_DF.to_csv(
    TABLE_DIR / "test_calibration_metrics_percent.csv",
    index=False,
)

display(
    CALIBRATION_PERCENT_DF
)

                         model  accuracy  mean_confidence  \
0               Final Proposed  0.838990         0.806542   
1    Final Proposed + Val-Temp  0.838990         0.838379   
2                SimCLR No-TTA  0.844515         0.857030   
3              DINOv2 ViT-S/14  0.796369         0.800820   
4  Optimized Weighted Ensemble  0.850829         0.792491   

   confidence_accuracy_gap    ece_15  adaptive_ece_15    mce_15       nll  \
0                -0.032448  0.037216         0.034710  0.406083  0.462707   
1                -0.000611  0.019286         0.015423  0.109956  0.452861   
2                 0.012516  0.026642         0.015759  0.262310  0.466243   
3                 0.004451  0.019809         0.015468  0.239485  0.523484   
4                -0.058338  0.060813         0.058338  0.250463  0.436539   

      brier  
0  0.231797  
1  0.229819  
2  0.227361  
3  0.282685  
4  0.223470  

                         model   accuracy  mean_confidence  \
0               Final Proposed  83.898974        80.654190   
1    Final Proposed + Val-Temp  83.898974        83.837918   
2                SimCLR No-TTA  84.451460        85.703021   
3              DINOv2 ViT-S/14  79.636938        80.082037   
4  Optimized Weighted Ensemble  85.082873        79.249060   

   confidence_accuracy_gap    ece_15  adaptive_ece_15     mce_15       nll  \
0                -3.244784  3.721619         3.470991  40.608344  0.462707   
1                -0.061056  1.928575         1.542334  10.995579  0.452861   
2                 1.251561  2.664236         1.575854  26.230952  0.466243   
3                 0.445100  1.980929         1.546829  23.948495  0.523484   
4                -5.833813  6.081303         5.833813  25.046276  0.436539   

      brier  
0  0.231797  
1  0.229819  
2  0.227361  
3  0.282685  
4  0.223470  

In [8]:
# ============================================================
# CELL 8 — RELIABILITY BIN TABLES
# ============================================================

for model_name, prob in MODEL_PROB.items():
    _, _, rel = ece_equal_width(
        Y_TEST,
        prob,
        n_bins=N_BINS,
    )

    safe_name = (
        model_name
        .lower()
        .replace(" ", "_")
        .replace("+", "plus")
        .replace("/", "_")
    )

    rel.to_csv(
        TABLE_DIR
        / f"reliability_bins_{safe_name}.csv",
        index=False,
    )

print("Reliability-bin tables saved.")

Reliability-bin tables saved.


In [9]:
# ============================================================
# CELL 9 — RELIABILITY DIAGRAM: FINAL RAW VS VALIDATION-TEMP
# ============================================================

plt.figure(
    figsize=(7.3, 6.2)
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect calibration",
)

for model_name, prob in [
    ("Final Proposed", FINAL_TEST),
    ("Final + Val-Temp", FINAL_TEST_TEMP),
]:
    _, _, rel = ece_equal_width(
        Y_TEST,
        prob,
        n_bins=N_BINS,
    )

    rel = rel.loc[
        rel["n"] > 0
    ]

    plt.plot(
        rel["mean_confidence"],
        rel["accuracy"],
        marker="o",
        label=model_name,
    )

plt.xlabel(
    "Mean predicted confidence"
)

plt.ylabel(
    "Observed accuracy"
)

plt.title(
    "Reliability Diagram — Final Proposed Model"
)

plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()

plt.savefig(
    FIG_DIR / "final_reliability_raw_vs_temperature_scaled.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [10]:
# ============================================================
# CELL 10 — RELIABILITY DIAGRAM: MAIN MODEL COMPARISON
# ============================================================

plt.figure(
    figsize=(7.8, 6.4)
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect calibration",
)

for model_name, prob in [
    ("Final Proposed", FINAL_TEST),
    ("SimCLR", SIMCLR_TEST),
    ("DINOv2", DINO_TEST),
    ("Weighted Ensemble", WEIGHTED_TEST),
]:
    _, _, rel = ece_equal_width(
        Y_TEST,
        prob,
        n_bins=N_BINS,
    )

    rel = rel.loc[
        rel["n"] > 0
    ]

    plt.plot(
        rel["mean_confidence"],
        rel["accuracy"],
        marker="o",
        label=model_name,
    )

plt.xlabel(
    "Mean predicted confidence"
)

plt.ylabel(
    "Observed accuracy"
)

plt.title(
    "Reliability Comparison — Original Internal Test"
)

plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()

plt.savefig(
    FIG_DIR / "main_models_reliability_comparison.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [11]:
# ============================================================
# CELL 11 — SELECTIVE PREDICTION HELPERS
# ============================================================

def fixed_class_balanced_accuracy(
    y_true,
    y_pred,
):
    recalls = []
    present = []

    for c in range(NUM_CLASSES):
        mask = (
            y_true == c
        )

        n = int(
            np.sum(mask)
        )

        present.append(n)

        if n == 0:
            recalls.append(
                np.nan
            )
        else:
            recalls.append(
                float(
                    np.mean(
                        y_pred[mask] == c
                    )
                )
            )

    valid = np.isfinite(
        recalls
    )

    bacc = (
        float(
            np.mean(
                np.asarray(recalls)[valid]
            )
        )
        if np.any(valid)
        else np.nan
    )

    return bacc, recalls, present

def selective_table(
    y_true,
    probabilities,
    coverages=COVERAGES,
):
    p = normalize_probabilities(
        probabilities
    )

    pred = np.argmax(
        p,
        axis=1,
    )

    conf = np.max(
        p,
        axis=1,
    )

    order = np.argsort(
        -conf
    )

    rows = []

    for coverage in coverages:
        n_keep = int(
            math.ceil(
                coverage
                * len(y_true)
            )
        )

        idx = order[
            :n_keep
        ]

        yt = y_true[
            idx
        ]

        yp = pred[
            idx
        ]

        selected_conf = conf[
            idx
        ]

        threshold = float(
            selected_conf.min()
        )

        acc = float(
            accuracy_score(
                yt,
                yp,
            )
        )

        bacc, recalls, class_counts = (
            fixed_class_balanced_accuracy(
                yt,
                yp,
            )
        )

        macro_f1 = float(
            f1_score(
                yt,
                yp,
                labels=np.arange(NUM_CLASSES),
                average="macro",
                zero_division=0,
            )
        )

        row = {
            "coverage": coverage,
            "retained_n": n_keep,
            "confidence_threshold": threshold,
            "accuracy": acc,
            "balanced_accuracy_present_classes": bacc,
            "macro_f1_fixed_8class": macro_f1,
            "error_risk": 1.0 - acc,
            "classes_present": int(
                np.sum(
                    np.asarray(class_counts) > 0
                )
            ),
        }

        for c, class_name in enumerate(
            CLASS_NAMES
        ):
            row[
                f"retained_{class_name}"
            ] = int(
                class_counts[c]
            )

        rows.append(row)

    return pd.DataFrame(
        rows
    )

print("Selective-prediction helpers ready.")

Selective-prediction helpers ready.


In [12]:
# ============================================================
# CELL 12 — SELECTIVE COVERAGE TABLES
# ============================================================

SELECTIVE_TABLES = {}

for model_name, prob in [
    ("Final Proposed", FINAL_TEST),
    ("Final Proposed + Val-Temp", FINAL_TEST_TEMP),
    ("SimCLR No-TTA", SIMCLR_TEST),
    ("DINOv2 ViT-S/14", DINO_TEST),
]:
    table = selective_table(
        Y_TEST,
        prob,
    )

    SELECTIVE_TABLES[
        model_name
    ] = table

    safe_name = (
        model_name
        .lower()
        .replace(" ", "_")
        .replace("+", "plus")
        .replace("/", "_")
    )

    table.to_csv(
        TABLE_DIR
        / f"selective_prediction_{safe_name}.csv",
        index=False,
    )

print("Final proposed selective prediction:")
display(
    SELECTIVE_TABLES[
        "Final Proposed"
    ]
)

print("\nFinal + validation-temperature selective prediction:")
display(
    SELECTIVE_TABLES[
        "Final Proposed + Val-Temp"
    ]
)

Final proposed selective prediction:


   coverage  retained_n  confidence_threshold  accuracy  \
0       1.0        2534              0.253156  0.838990   
1       0.9        2281              0.512233  0.881192   
2       0.8        2028              0.630914  0.922091   
3       0.7        1774              0.730628  0.940812   
4       0.6        1521              0.816776  0.959895   
5       0.5        1267              0.876115  0.974743   

   balanced_accuracy_present_classes  macro_f1_fixed_8class  error_risk  \
0                           0.839549               0.797961    0.161010   
1                           0.883014               0.861355    0.118808   
2                           0.917062               0.911540    0.077909   
3                           0.940993               0.936946    0.059188   
4                           0.955320               0.954619    0.040105   
5                           0.963251               0.967234    0.025257   

   classes_present  retained_MEL  retained_NV  retained_BCC 


Final + validation-temperature selective prediction:


   coverage  retained_n  confidence_threshold  accuracy  \
0       1.0        2534              0.269874  0.838990   
1       0.9        2281              0.550010  0.881631   
2       0.8        2028              0.677750  0.921598   
3       0.7        1774              0.784987  0.939121   
4       0.6        1521              0.865199  0.959895   
5       0.5        1267              0.916663  0.974743   

   balanced_accuracy_present_classes  macro_f1_fixed_8class  error_risk  \
0                           0.839549               0.797961    0.161010   
1                           0.883344               0.857077    0.118369   
2                           0.916509               0.911756    0.078402   
3                           0.939225               0.933892    0.060879   
4                           0.956104               0.955156    0.040105   
5                           0.963113               0.967036    0.025257   

   classes_present  retained_MEL  retained_NV  retained_BCC 

In [13]:
# ============================================================
# CELL 13 — RISK-COVERAGE CURVE + AURC
# ============================================================

def risk_coverage_curve(
    y_true,
    probabilities,
):
    p = normalize_probabilities(
        probabilities
    )

    pred = np.argmax(
        p,
        axis=1,
    )

    conf = np.max(
        p,
        axis=1,
    )

    correct = (
        pred == y_true
    ).astype(np.float64)

    order = np.argsort(
        -conf
    )

    sorted_correct = correct[
        order
    ]

    cumulative_errors = np.cumsum(
        1.0
        - sorted_correct
    )

    k = np.arange(
        1,
        len(y_true) + 1,
    )

    coverage = (
        k
        / len(y_true)
    )

    risk = (
        cumulative_errors
        / k
    )

    aurc = float(
        np.trapz(
            risk,
            coverage,
        )
    )

    return coverage, risk, aurc

risk_rows = []

plt.figure(
    figsize=(8.0, 5.8)
)

for model_name, prob in [
    ("Final Proposed", FINAL_TEST),
    ("Final + Val-Temp", FINAL_TEST_TEMP),
    ("SimCLR", SIMCLR_TEST),
    ("DINOv2", DINO_TEST),
]:
    coverage, risk, aurc = risk_coverage_curve(
        Y_TEST,
        prob,
    )

    risk_rows.append({
        "model": model_name,
        "aurc": aurc,
    })

    # plot from 30% coverage onward for readability
    mask = (
        coverage >= 0.30
    )

    plt.plot(
        100 * coverage[mask],
        100 * risk[mask],
        label=f"{model_name} (AURC={aurc:.4f})",
    )

RISK_DF = pd.DataFrame(
    risk_rows
)

display(
    RISK_DF
)

RISK_DF.to_csv(
    TABLE_DIR / "risk_coverage_aurc.csv",
    index=False,
)

plt.xlabel(
    "Retained coverage (%)"
)

plt.ylabel(
    "Error risk (%)"
)

plt.title(
    "Risk-Coverage Curves — Original Internal Test"
)

plt.grid(alpha=0.2)
plt.legend()
plt.tight_layout()

plt.savefig(
    FIG_DIR / "risk_coverage_curves.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

/tmp/ipykernel_16/4113661974.py:56: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  np.trapz(


              model      aurc
0    Final Proposed  0.041649
1  Final + Val-Temp  0.041615
2            SimCLR  0.041262
3            DINOv2  0.064201

In [14]:
# ============================================================
# CELL 14 — SELECTIVE BACC / ACCURACY FIGURE
# ============================================================

final_sel = SELECTIVE_TABLES[
    "Final Proposed"
]

plt.figure(
    figsize=(7.8, 5.6)
)

plt.plot(
    100 * final_sel["coverage"],
    100 * final_sel["accuracy"],
    marker="o",
    label="Accuracy",
)

plt.plot(
    100 * final_sel["coverage"],
    100 * final_sel[
        "balanced_accuracy_present_classes"
    ],
    marker="o",
    label="Balanced Accuracy",
)

plt.gca().invert_xaxis()

plt.xlabel(
    "Retained coverage (%)"
)

plt.ylabel(
    "Performance (%)"
)

plt.title(
    "Final Proposed Model — Selective Prediction"
)

plt.grid(alpha=0.2)
plt.legend()
plt.tight_layout()

plt.savefig(
    FIG_DIR / "final_selective_prediction_performance.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [15]:
# ============================================================
# CELL 15 — JOURNAL SUMMARY / AUDIT
# ============================================================

raw_metrics = calibration_metrics(
    Y_TEST,
    FINAL_TEST,
)

temp_metrics = calibration_metrics(
    Y_TEST,
    FINAL_TEST_TEMP,
)

summary = {
    "protocol": "original stratified 80/10/10",
    "validation_n": len(Y_VAL),
    "test_n": len(Y_TEST),
    "final_method": "NoTTA_StackingBlend_ClassBias",
    "test_used_for_model_selection": False,
    "temperature_scaling": temperature_audit,
    "final_raw_calibration": raw_metrics,
    "final_temperature_scaled_calibration": temp_metrics,
    "temperature_scaling_changes_argmax": False,
    "calibration_models": CALIBRATION_DF.to_dict(
        orient="records"
    ),
    "aurc": RISK_DF.to_dict(
        orient="records"
    ),
    "interpretation_rule": (
        "Calibration and selective-prediction analyses are descriptive "
        "reliability analyses on the frozen held-out test set. "
        "The optional scalar temperature is selected only on validation "
        "and does not change the final model's class predictions."
    ),
}

with open(
    AUDIT_DIR / "J6_calibration_selective_summary.json",
    "w",
) as f:
    json.dump(
        summary,
        f,
        indent=2,
        default=float,
    )

print(
    json.dumps(
        summary,
        indent=2,
        default=float,
    )
)

{
  "protocol": "original stratified 80/10/10",
  "validation_n": 2533,
  "test_n": 2534,
  "final_method": "NoTTA_StackingBlend_ClassBias",
  "test_used_for_model_selection": false,
  "temperature_scaling": {
    "selection_split": "validation only",
    "test_used_for_temperature_selection": false,
    "selected_temperature": 0.866,
    "raw_validation_nll": 0.4789775446151375,
    "scaled_validation_nll": 0.47181396066315745,
    "test_argmax_preserved": true
  },
  "final_raw_calibration": {
    "accuracy": 0.8389897395422258,
    "mean_confidence": 0.8065419041702859,
    "confidence_accuracy_gap": -0.03244783537193985,
    "ece_15": 0.03721619496011349,
    "adaptive_ece_15": 0.03470991245018694,
    "mce_15": 0.40608344266648766,
    "nll": 0.4627067132189287,
    "brier": 0.23179707670698838
  },
  "final_temperature_scaled_calibration": {
    "accuracy": 0.8389897395422258,
    "mean_confidence": 0.8383791767055059,
    "confidence_accuracy_gap": -0.0006105628367198968,
    "e

In [16]:
# ============================================================
# CELL 16 — COMPLETION MARKER + ZIP
# ============================================================

marker = {
    "stage": "J6_original_split_calibration_selective_prediction",
    "status": "complete",
    "protocol": "original_stratified_80_10_10",
    "validation_n": len(Y_VAL),
    "test_n": len(Y_TEST),
    "selected_temperature": BEST_TEMPERATURE,
    "test_used_for_temperature_selection": False,
    "final_raw_ece": raw_metrics["ece_15"],
    "final_scaled_ece": temp_metrics["ece_15"],
    "final_raw_nll": raw_metrics["nll"],
    "final_scaled_nll": temp_metrics["nll"],
    "output_rel": "j6_original_split_calibration",
}

MARKER_PATH = (
    WORKING
    / "J6_CALIBRATION_COMPLETE.json"
)

with open(
    MARKER_PATH,
    "w",
) as f:
    json.dump(
        marker,
        f,
        indent=2,
    )

shutil.copy2(
    MARKER_PATH,
    OUTPUT_DIR
    / "J6_CALIBRATION_COMPLETE.json",
)

ZIP_PATH = (
    WORKING
    / "J6_ORIGINAL_SPLIT_CALIBRATION_RESULTS.zip"
)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:
    for p in sorted(
        OUTPUT_DIR.rglob("*")
    ):
        if p.is_file():
            zf.write(
                p,
                arcname=str(
                    p.relative_to(
                        OUTPUT_DIR.parent
                    )
                ),
            )

required = [
    TABLE_DIR / "test_calibration_metrics.csv",
    TABLE_DIR / "risk_coverage_aurc.csv",
    TABLE_DIR / "selective_prediction_final_proposed.csv",
    FIG_DIR / "final_reliability_raw_vs_temperature_scaled.png",
    FIG_DIR / "main_models_reliability_comparison.png",
    FIG_DIR / "risk_coverage_curves.png",
    FIG_DIR / "final_selective_prediction_performance.png",
    AUDIT_DIR / "J6_calibration_selective_summary.json",
    MARKER_PATH,
    ZIP_PATH,
]

missing = [
    str(p)
    for p in required
    if not p.exists()
]

if missing:
    raise FileNotFoundError(
        "Missing J6 outputs:\n"
        + "\n".join(
            missing
        )
    )

print("=" * 96)
print("J6 CALIBRATION / SELECTIVE PREDICTION COMPLETE")
print("=" * 96)

print(
    "Validation-selected temperature:",
    BEST_TEMPERATURE
)

print(
    "Final raw ECE   :",
    f"{100 * raw_metrics['ece_15']:.3f}%"
)

print(
    "Final scaled ECE:",
    f"{100 * temp_metrics['ece_15']:.3f}%"
)

print(
    "Final raw NLL   :",
    f"{raw_metrics['nll']:.5f}"
)

print(
    "Final scaled NLL:",
    f"{temp_metrics['nll']:.5f}"
)

print(
    "ZIP:",
    ZIP_PATH
)

print()
print(
    "Save this Kaggle notebook Version WITH OUTPUTS "
    "and send the executed notebook back."
)

J6 CALIBRATION / SELECTIVE PREDICTION COMPLETE
Validation-selected temperature: 0.866
Final raw ECE   : 3.722%
Final scaled ECE: 1.929%
Final raw NLL   : 0.46271
Final scaled NLL: 0.45286
ZIP: /kaggle/working/J6_ORIGINAL_SPLIT_CALIBRATION_RESULTS.zip

Save this Kaggle notebook Version WITH OUTPUTS and send the executed notebook back.
